# Pilot V1 - Single Telecom Tower Fragility

This notebook is the first beginner-friendly fragility prototype.

In simple words: we take one 48 m steel lattice telecom tower, increase the wind speed step by step, run small random simulations at each wind speed, count how often collapse happens, and fit a smooth fragility curve.

This is not an OpenSeesPy model yet. It is a clean MVP that shows the workflow before replacing the simple response model with a structural model.

## 1. Imports and Notebook Settings

This cell loads the scientific Python libraries and sets up paths. The output folder is always created relative to the repository, so the notebook works even if someone downloads the project on a different computer.

In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

CACHE_DIR = REPO_ROOT / 'outputs' / '_cache'
MPL_CACHE_DIR = REPO_ROOT / 'outputs' / '_matplotlib_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('XDG_CACHE_HOME', str(CACHE_DIR))
os.environ.setdefault('MPLCONFIGDIR', str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 130
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

OUTPUT_DIR = REPO_ROOT / 'outputs' / 'notebook_v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Repository root: {REPO_ROOT}')
print(f'Notebook outputs: {OUTPUT_DIR}')

## 2. Literature-Based Pilot Assumptions

These are the simple assumptions for the first pilot: one self-supporting steel lattice telecom tower, wind-only hazard, 10-minute mean wind speed, collapse-only damage state, and wind stripes from 20 to 50 m/s.

In [ ]:
wind_speed_stripes_mps = np.round(np.arange(20.0, 50.0 + 0.1, 2.5), 2)

assumptions = {
    'project_name': 'Notebook Pilot V1 single-tower telecom fragility under wind',
    'tower_class': 'self-supporting steel lattice telecommunication tower',
    'shape': 'square lattice',
    'structural_height_m': 48.0,
    'total_height_including_tip_m': 51.0,
    'lower_inclined_height_m': 24.0,
    'upper_straight_height_m': 24.0,
    'hazard': 'wind only',
    'intensity_measure': '10-minute mean wind speed in m/s',
    'wind_speed_stripes_mps': wind_speed_stripes_mps.tolist(),
    'wind_directions_deg': [0.0],
    'simulations_per_stripe': 12,
    'damage_state': 'collapse only',
    'uncertainty': {
        'wind_load_multiplier_cov': 0.20,
        'capacity_multiplier_cov': 0.15,
    },
    'pilot_response_model': {
        'reference_collapse_speed_mps': 38.0,
        'base_capacity_index': 1.0,
    },
}

assumptions

## 3. Uncertainty Model

Real towers and real wind loads are not perfectly known. For the MVP we use simple lognormal multipliers:

- wind load multiplier: makes wind demand randomly a little higher or lower
- capacity multiplier: makes tower strength randomly a little higher or lower

These are pilot assumptions, not final calibrated research values.

In [ ]:
def lognormal_mu_sigma_from_mean_cov(mean_value, cov_value):
    '''Convert a lognormal mean and coefficient of variation into log-space parameters.'''
    if mean_value <= 0:
        raise ValueError('mean_value must be positive')
    if cov_value < 0:
        raise ValueError('cov_value cannot be negative')

    sigma = np.sqrt(np.log(1.0 + cov_value**2))
    mu = np.log(mean_value) - 0.5 * sigma**2
    return mu, sigma


def sample_lognormal_multiplier(mean_value, cov_value, size):
    '''Sample positive random multipliers from a lognormal distribution.'''
    mu, sigma = lognormal_mu_sigma_from_mean_cov(mean_value, cov_value)
    return rng.lognormal(mean=mu, sigma=sigma, size=size)


sample_lognormal_multiplier(mean_value=1.0, cov_value=0.20, size=5)

## 4. Simple Surrogate Response Model

Before OpenSeesPy, we use a simple demand-capacity idea:

- demand grows roughly with wind pressure, which grows with wind speed squared
- capacity is the tower resistance
- collapse happens when demand is greater than capacity

Later, this function is the part we replace with a real structural model.

In [ ]:
def compute_base_wind_effect(wind_speed_mps, tower_height_m=48.0):
    '''Compute a dimensionless wind demand index for the pilot model.'''
    if wind_speed_mps <= 0:
        raise ValueError('wind_speed_mps must be positive')
    if tower_height_m <= 0:
        raise ValueError('tower_height_m must be positive')

    reference_speed = assumptions['pilot_response_model']['reference_collapse_speed_mps']
    dynamic_pressure_ratio = (wind_speed_mps / reference_speed) ** 2
    height_factor = (tower_height_m / 48.0) ** 1.15
    return dynamic_pressure_ratio * height_factor


def compute_base_capacity(tower_height_m=48.0):
    '''Compute a dimensionless tower capacity index for the pilot model.'''
    if tower_height_m <= 0:
        raise ValueError('tower_height_m must be positive')

    base_capacity = assumptions['pilot_response_model']['base_capacity_index']
    height_factor = (48.0 / tower_height_m) ** 0.20
    return base_capacity * height_factor


def simulate_one_response(wind_speed_mps, tower_height_m, wind_multiplier, capacity_multiplier):
    '''Run one demand-capacity simulation and return collapse information.'''
    demand = compute_base_wind_effect(wind_speed_mps, tower_height_m) * wind_multiplier
    capacity = compute_base_capacity(tower_height_m) * capacity_multiplier
    collapse = demand > capacity

    return {
        'demand': float(demand),
        'capacity': float(capacity),
        'collapse': bool(collapse),
    }


simulate_one_response(38.0, 48.0, wind_multiplier=1.0, capacity_multiplier=1.0)

## 5. Run Wind-Speed Stripe Analysis

For each wind speed stripe, we run 12 simulations. Then we count how many simulations collapsed. For example, if 3 out of 12 collapse, the observed collapse probability is 0.25.

In [ ]:
stripe_records = []

for wind_speed_mps in assumptions['wind_speed_stripes_mps']:
    wind_multipliers = sample_lognormal_multiplier(1.0, 0.20, assumptions['simulations_per_stripe'])
    capacity_multipliers = sample_lognormal_multiplier(1.0, 0.15, assumptions['simulations_per_stripe'])

    demand_values = []
    capacity_values = []
    collapse_flags = []

    for wind_multiplier, capacity_multiplier in zip(wind_multipliers, capacity_multipliers):
        result = simulate_one_response(
            wind_speed_mps=wind_speed_mps,
            tower_height_m=assumptions['structural_height_m'],
            wind_multiplier=wind_multiplier,
            capacity_multiplier=capacity_multiplier,
        )
        demand_values.append(result['demand'])
        capacity_values.append(result['capacity'])
        collapse_flags.append(int(result['collapse']))

    failure_count = int(np.sum(collapse_flags))
    total_count = assumptions['simulations_per_stripe']

    stripe_records.append({
        'wind_speed_mps': float(wind_speed_mps),
        'simulations_in_stripe': total_count,
        'failure_count': failure_count,
        'observed_collapse_probability': failure_count / total_count,
        'mean_demand': float(np.mean(demand_values)),
        'mean_capacity': float(np.mean(capacity_values)),
    })

stripe_results_df = pd.DataFrame(stripe_records)
stripe_results_df

## 6. Fit a Lognormal Fragility Curve

A fragility curve gives the probability of collapse at a wind speed. We fit this form:

`P(collapse | V) = Phi((ln(V) - ln(theta)) / beta)`

`theta` is the median collapse wind speed. `beta` controls how spread out the curve is.

In [ ]:
def lognormal_fragility_probability(wind_speed_mps, theta, beta):
    '''Compute collapse probability from a lognormal fragility curve.'''
    wind_speed_mps = np.asarray(wind_speed_mps, dtype=float)
    probability = norm.cdf((np.log(wind_speed_mps) - np.log(theta)) / beta)
    return np.clip(probability, 1e-10, 1.0 - 1e-10)


def negative_log_likelihood(log_parameters, wind_speeds, failures, totals):
    '''Binomial negative log-likelihood for stripe data.'''
    theta = np.exp(log_parameters[0])
    beta = np.exp(log_parameters[1])
    p_fail = lognormal_fragility_probability(wind_speeds, theta, beta)
    log_likelihood = failures * np.log(p_fail) + (totals - failures) * np.log(1.0 - p_fail)
    return -float(np.sum(log_likelihood))


wind_speeds = stripe_results_df['wind_speed_mps'].to_numpy(dtype=float)
failures = stripe_results_df['failure_count'].to_numpy(dtype=float)
totals = stripe_results_df['simulations_in_stripe'].to_numpy(dtype=float)

fit_result = minimize(
    negative_log_likelihood,
    x0=np.log([38.0, 0.20]),
    args=(wind_speeds, failures, totals),
    method='L-BFGS-B',
    bounds=[(np.log(1.0), np.log(200.0)), (np.log(0.03), np.log(2.0))],
)

if not fit_result.success:
    raise RuntimeError(f'Fragility fitting failed: {fit_result.message}')

theta_mps = float(np.exp(fit_result.x[0]))
beta = float(np.exp(fit_result.x[1]))

fragility_summary = {
    'theta_mps': theta_mps,
    'beta': beta,
    'negative_log_likelihood': float(fit_result.fun),
    'interpretation': 'theta is the wind speed where collapse probability is about 50 percent',
}

fragility_summary

## 7. Plot the Fragility Curve

The black dots are the observed collapse probabilities from the 12 simulations per stripe. The red line is the fitted smooth lognormal fragility curve.

In [ ]:
smooth_wind_speeds = np.linspace(wind_speeds.min(), wind_speeds.max(), 300)
smooth_probabilities = lognormal_fragility_probability(smooth_wind_speeds, theta_mps, beta)

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.scatter(
    stripe_results_df['wind_speed_mps'],
    stripe_results_df['observed_collapse_probability'],
    color='black',
    s=70,
    label='Observed stripe results',
    zorder=3,
)
ax.plot(smooth_wind_speeds, smooth_probabilities, color='tab:red', linewidth=2.5, label='Fitted lognormal curve')
ax.axvline(theta_mps, color='tab:blue', linestyle='--', label=f'theta = {theta_mps:.2f} m/s')
ax.set_xlabel('10-minute mean wind speed, V (m/s)')
ax.set_ylabel('Probability of collapse')
ax.set_title('Notebook Pilot V1 fragility curve for one 48 m telecom tower')
ax.set_ylim(-0.02, 1.02)
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 8. Save Outputs

This cell saves the assumptions, stripe results, fragility summary, and plot data. The notebook remains visual, but the CSV and JSON files make the results easy to reuse later.

In [ ]:
stripe_results_df['fitted_collapse_probability'] = lognormal_fragility_probability(
    stripe_results_df['wind_speed_mps'].to_numpy(dtype=float),
    theta_mps,
    beta,
)

with (OUTPUT_DIR / 'assumptions.json').open('w', encoding='utf-8') as file:
    json.dump(assumptions, file, indent=4)

stripe_results_df.to_csv(OUTPUT_DIR / 'stripe_results.csv', index=False)

with (OUTPUT_DIR / 'fragility_summary.json').open('w', encoding='utf-8') as file:
    json.dump(fragility_summary, file, indent=4)

print(f'Saved notebook V1 outputs to: {OUTPUT_DIR}')
print(f'Median collapse wind speed theta: {theta_mps:.2f} m/s')
print(f'Lognormal beta: {beta:.3f}')